In [2]:
!pip uninstall -y xgboost
!pip install xgboost==2.0.3

Found existing installation: xgboost 3.0.5
Uninstalling xgboost-3.0.5:
  Successfully uninstalled xgboost-3.0.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.1/297.1 MB 4.9 MB/s eta 0:00:00


In [3]:
!pip install scikit-learn==1.3.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
imbalanced-learn 0.14.0 requires scikit-learn<2,>=1.4.2, but you have scikit-learn 1.3.2 which is incompatible.
opencv-contrib-pyt

In [5]:
# ============================================
# 다사읍 도로 관련 피처 변경 시뮬레이션 (fire_time)
# - 학습 파이프라인 로드
# - fire_time_pred.csv에서 학습 피처 목록 복원(차원 불일치 방지)
# - baseline 예측 → 다사읍의 road_len_4/road_len_5.5/road_len/ratio_road_len_4 수정 → 수정 예측
# - time_hour / (있으면) count_hour 모두 비교 저장
# ============================================

import os, joblib
import numpy as np
import pandas as pd

# -----------------------------
# 0) 경로/설정
# -----------------------------
SAVE_DIR = "/content/drive/MyDrive/대구빅데이터분석/화재"       # 학습 산출물 폴더
PRED_CSV = f"{SAVE_DIR}/fire_time_pred.csv"                     # 학습 당시 예측결과(피처 복원용)

# 모델(둘 다 있으면 둘 다 예측, 없으면 있는 것만)
MODEL_TIME = f"{SAVE_DIR}/model_fire_time_time_hour.joblib"
MODEL_COUNT = f"{SAVE_DIR}/model_fire_time_count_hour.joblib"   # 없을 수도 있음

# 원천 데이터
DAEGU_INFO_CSV = "/content/drive/MyDrive/대구빅데이터분석/대구통합정보.csv"
FIRE_TIME_CSV  = "/content/drive/MyDrive/대구빅데이터분석/화재/화재_시간.csv"

# 시뮬레이션 대상
DONG_NAME = "다사읍"

# 바꾸고 싶은 값 지정
EDIT_VALUES = {
    "road_len_4":       10.40206311,   # 예: 4m 미만 도로길이(km)
    "road_len_5.5":     34.22991378,   # 예: 4~5.5m 도로길이(km)  ← 컬럼명에 점(.) 포함 주의
    "ratio_road_len_4": 0.0690190187,  # 예: 4m 미만 도로 비율(0~1)
}

OUT_CSV = f"{SAVE_DIR}/sim_dasa_road_timehour.csv"

TARGETS = ["count_hour", "time_hour"]  # fire_time 타깃

# -----------------------------
# 1) 유틸
# -----------------------------
def load_model(path: str):
    if not os.path.exists(path):
        return None
    return joblib.load(path)  # sklearn Pipeline

def load_dong_mapping_from_pred(pred_csv: str) -> dict:
    dfp = pd.read_csv(pred_csv)
    if not {"dong","dong_le"} <= set(dfp.columns):
        raise ValueError("pred csv에 'dong','dong_le'가 없습니다.")
    return dict(dfp[["dong","dong_le"]].drop_duplicates().values.tolist())

def get_trained_features_from_predcsv(pred_csv: str, targets: list[str]) -> list[str]:
    dfp = pd.read_csv(pred_csv)
    cols = []
    for c in dfp.columns:
        if c in targets:                # 타깃 제외
            continue
        if c.startswith("pred_") or c.startswith("oof_"):  # 예측/OOF 제외
            continue
        cols.append(c)
    return cols

def attach_dong_le(df: pd.DataFrame, dong_map: dict) -> pd.DataFrame:
    out = df.copy()
    if "dong_le" not in out.columns:
        out["dong_le"] = out["dong"].map(dong_map).fillna(-1).astype(int)
    return out

def predict_target(pipe_model, X: pd.DataFrame) -> np.ndarray:
    # 학습이 log1p(target) → 예측 후 expm1 + 음수 클립
    y_hat = np.expm1(pipe_model.predict(X))
    return np.clip(y_hat, a_min=0, a_max=None)

def merge_with_base_first(base, other, on='dong', how='left'):
    dup = set(base.columns).intersection(other.columns) - {on}
    other_wo = other.drop(columns=list(dup))
    merged = base.merge(other_wo, on=on, how=how)
    base_cols = list(base.columns)
    extra_cols = [c for c in merged.columns if c not in base_cols]
    return merged[base_cols + extra_cols]

# -----------------------------
# 2) 원천 데이터 로드 & 한→영 매핑(학습과 동일)
# -----------------------------
daegu_info = pd.read_csv(DAEGU_INFO_CSV)
fire_time  = pd.read_csv(FIRE_TIME_CSV)

daegu_info_colmap = {
    "시군구":"sigungu","읍면동":"dong","인구(명)":"population","면적(㎢)":"area","세대수":"household",
    "인구밀도(인/㎢)":"pop_density","평균 세대당 인수":"household_size","자가주차장확보율":"parking",
    "전통시장 수":"market","일일차량 통행량":"traffic","일일차량 평균속도":"speed",
    "4m미만_도로폭(km)":"road_len_4","4~5.5m_도로폭(km)":"road_len_5.5","5.5~8m_도로폭(km)":"road_len_8",
    "8~12m_도로폭(km)":"road_len_12","12m이상_도로폭(km)":"road_len_12up","전체도로길이(km)":"road_len",
    "4m미만_도로폭비율":"ratio_road_len_4",
    "연평균구급출동거리":"ambulance_len","연평균구급출동횟수":"ambulance_count",
    "연평균구조출동거리":"rescue_len","연평균구조출동횟수":"rescue_count",
    "연평균생활안전출동거리":"safety_len","연평균생활안전출동횟수":"safety_count",
    "연평균화재출동거리":"fire_len","연평균화재출동횟수":"fire_count"
}
fire_hour_colmap = {"시군구":"sigungu","읍면동":"dong","시간":"hour","시간별출동횟수":"count_hour","시간별도착시간":"time_hour"}

daegu_info = daegu_info.rename(columns=daegu_info_colmap)
fire_time  = fire_time.rename(columns=fire_hour_colmap)

fire_time_merged = merge_with_base_first(daegu_info, fire_time, on='dong')

# -----------------------------
# 3) 모델/매핑/피처 복원
# -----------------------------
pipe_time  = load_model(MODEL_TIME)
pipe_count = load_model(MODEL_COUNT)  # 없으면 None

if pipe_time is None and pipe_count is None:
    raise FileNotFoundError("time_hour/count_hour 모델이 모두 없습니다.")

dong_map          = load_dong_mapping_from_pred(PRED_CSV)
trained_features  = get_trained_features_from_predcsv(PRED_CSV, TARGETS)

df0 = attach_dong_le(fire_time_merged, dong_map)

# 학습 피처에 'dong'이 들어있고 'dong_le'가 없다면 교정
if "dong" in trained_features and "dong_le" not in trained_features:
    trained_features = ["dong_le" if c == "dong" else c for c in trained_features]

# 현재 입력에 없는 학습 피처가 있는지 확인
missing = [c for c in trained_features if c not in df0.columns]
if missing:
    raise ValueError(f"입력 데이터에 없는 학습 피처가 있습니다: {missing}")

mask_dong = (df0["dong"] == DONG_NAME)
if mask_dong.sum() == 0:
    raise ValueError(f"입력에 '{DONG_NAME}'이 없습니다. dong 예시: {df0['dong'].unique()[:10]}")

# -----------------------------
# 4) Baseline 예측
# -----------------------------
X_base = df0[trained_features]
df_base = df0.copy()
if pipe_time is not None:
    df_base["pred_time_hour_base"] = predict_target(pipe_time, X_base)
if pipe_count is not None:
    df_base["pred_count_hour_base"] = predict_target(pipe_count, X_base)

# -----------------------------
# 5) 편집값 적용 → 수정 예측
# -----------------------------
df_mod = df0.copy()

# 존재하는 컬럼만 수정
for k, v in EDIT_VALUES.items():
    if k not in df_mod.columns:
        raise KeyError(f"'{k}' 컬럼이 데이터에 없습니다. 실제 컬럼들: {list(df_mod.columns)[:30]} ...")
    df_mod.loc[mask_dong, k] = v

X_mod = df_mod[trained_features]
if pipe_time is not None:
    df_mod["pred_time_hour_mod"] = predict_target(pipe_time, X_mod)
if pipe_count is not None:
    df_mod["pred_count_hour_mod"] = predict_target(pipe_count, X_mod)

# -----------------------------
# 6) 다사읍 전/후 비교 테이블
# -----------------------------
keep_cols = ["dong", "hour"] + list(EDIT_VALUES.keys())
keep_cols = [c for c in keep_cols if c in df_mod.columns]  # 안전 필터

cols_base = []
cols_mod  = []
if pipe_time is not None:
    cols_base.append("pred_time_hour_base")
    cols_mod.append("pred_time_hour_mod")
if pipe_count is not None:
    cols_base.append("pred_count_hour_base")
    cols_mod.append("pred_count_hour_mod")

comp = pd.concat([
    df_base.loc[mask_dong, keep_cols + cols_base].reset_index(drop=True),
    df_mod.loc[mask_dong, cols_mod].reset_index(drop=True)
], axis=1)

# 변화량/변화율(각 예측마다 계산)
for b, m in zip(cols_base, cols_mod):
    comp[f"diff_{m.replace('pred_','')}"] = comp[m] - comp[b]
    comp[f"pct_change_%_{m.replace('pred_','')}"] = np.where(
        comp[b] > 0, 100.0 * comp[f"diff_{m.replace('pred_','')}"] / comp[b], np.nan
    )

# 요약(평균)
summary_cols = []
for b, m in zip(cols_base, cols_mod):
    summary_cols += [b, m, f"diff_{m.replace('pred_','')}", f"pct_change_%_{m.replace('pred_','')}"]

summary = comp[summary_cols].mean().to_frame("mean").T

# -----------------------------
# 7) 저장 & 출력
# -----------------------------
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
comp.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print(f"\n[다사읍 도로 피처 시뮬레이션 저장] {OUT_CSV}")
print(f" - 적용 EDIT_VALUES: {EDIT_VALUES}")
print(f" - 학습 피처 개수: {len(trained_features)} (fire_time_pred.csv 기준)")
print("\n[요약 평균]")
print(summary.round(4))



[다사읍 도로 피처 시뮬레이션 저장] /content/drive/MyDrive/대구빅데이터분석/화재/sim_dasa_road_timehour.csv
 - 적용 EDIT_VALUES: {'road_len_4': 10.40206311, 'road_len_5.5': 34.22991378, 'ratio_road_len_4': 0.0690190187}
 - 학습 피처 개수: 28 (fire_time_pred.csv 기준)

[요약 평균]
      pred_time_hour_base  pred_time_hour_mod  diff_time_hour_mod  \
mean               1.5712               1.325             -0.2463   

      pct_change_%_time_hour_mod  pred_count_hour_base  pred_count_hour_mod  \
mean                  -24.784401                1.0141               1.0444   

      diff_count_hour_mod  pct_change_%_count_hour_mod  
mean               0.0303                       4.1217  
